# Inherent Structures: Langevin to Quench
This notebook runs the temperature experiment directly from Colab: equilibrate with full-batch Langevin dynamics, quench with full-batch gradient descent, and measure the loss and overlap at the inherent structure.

In [ ]:
# Run this cell only when starting from a fresh Colab runtime.
# Replace REPO_URL with the repository URL or mount the project from Drive.
REPO_URL = "https://github.com/marjankowskam/Comparing_Dynamics.git"
!git clone {REPO_URL}
%cd Comparing_Dynamics
%pip install -r requirements.txt
%pip install -e .

In [ ]:
from pathlib import Path
import torch
import wandb

from comparing_dynamics.config import Config
from comparing_dynamics.experiments.inherent_structures import (
    InherentStructureConfig,
    run_inherent_structure,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
output_dir = Path("runs/inherent-structures")

In [ ]:
cfg = Config(
    initial_seed=19,
    task_name="fashion_mnist",
    task_params={"batch_size": None, "data_dir": "data"},
    model_name="mnistMLP",
    model_params={
        "hidden_dim": [100, 100, 10],
        "activation_fn": "relu",
    },
)

equilibration_steps = 1000
quench_steps = 5000
langevin_lr = 1e-3
quench_lr = 1e-2
temperatures = [0.0, 0.01, 0.1, 1.0]

In [ ]:
results = []

for temperature in temperatures:
    cfg.initial_seed = 19
    wb_run = wandb.init(
        project="comparing-dynamics",
        name=f"fashion-mnist-T-{temperature:g}",
        config={
            "temperature": temperature,
            "equilibration_steps": equilibration_steps,
            "quench_steps": quench_steps,
            "langevin_lr": langevin_lr,
            "quench_lr": quench_lr,
            "device": device,
        },
    )
    try:
        experiment_cfg = InherentStructureConfig(
            temperature=temperature,
            equilibration_steps=equilibration_steps,
            quench_steps=quench_steps,
            langevin_lr=langevin_lr,
            quench_lr=quench_lr,
            seed=cfg.initial_seed,
        )
        result = run_inherent_structure(
            cfg,
            experiment_cfg,
            device=device,
            save_dir=output_dir,
            run_name=wb_run.name,
        )
        wandb.log({
            "inherent_loss": result["final_loss"],
            "inherent_gradient_norm": result["final_gradient_norm"],
            "overlap_q": result["overlap_q"],
            "initial_to_inherent_overlap": result["initial_to_inherent_overlap"],
        })
        results.append(result)
    finally:
        wandb.finish()

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "temperature": result["temperature"],
        "inherent_loss": result["final_loss"],
        "gradient_norm": result["final_gradient_norm"],
        "overlap_q": result["overlap_q"],
        "initial_to_inherent_overlap": result["initial_to_inherent_overlap"],
    }
    for result in results
])
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(summary["temperature"], summary["inherent_loss"], marker="o")
axes[0].set(xlabel="Temperature T", ylabel="Inherent-structure loss")
axes[1].plot(summary["temperature"], summary["overlap_q"], marker="o")
axes[1].set(xlabel="Temperature T", ylabel="q(T)")
fig.tight_layout()

# SGD to Quench

In [ ]:
import torch 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl 

ModuleNotFoundError: No module named 'pandas'

In [ ]:
base_path = Path(base_path)
base_path="G:/My Drive/STUDIA/PhD/ICML_2026_Hessian_Eigenvectors/"

# Load some SGD run names
file_path = "../data/Comparing_Dynamics.xlsx"
df = pd.read_excel(file_path, sheet_name="ICML_runs")

# Load their wandb-saved metrics
file_path = "../data/Comparing_Dynamics_wandb_metrics.xlsx"
df_metrics = pd.read_excel(file_path, sheet_name="ICML_runs")

# Extract ones that have SGD. COnsider unique names only.
filtered = df[df["op_name"].isin(["SGD", "sgd"]) & (df["task_name"] == task_name)]
# filtered = filtered.drop_duplicates( subset=["opt_name", "batch size", "initial_seed", "opt_params.lr", "task_name"])
filtered = filtered.drop_duplicates( subset=["run_name"])

# For each name:
* Assign a temperature (accoring to each one of the metrics)

for run_name in filtered["run_name"].unique():
    cfg = Config.load_from_file(name, str(base_path)+"/")

    loss = df_metrics[df_metrics["run_name"] == run_name]["loss"].values[0]
    accuracy = df_metrics[df_metrics["run_name"] == run_name]["accuracy"].values[0]
    grad_norm = df_metrics[df_metrics["run_name"] == run_name]["grad_norm"].values[0]

    noise_scale_temperature = cfg["opt_params"]["lr"] / cfg["batch_size"]

    # control_temperature = get_control_temperature(cfg, metrics, params)
    # glassy_eff_temperature = get_glassy_effective_temperature(cfg, metrics, params)


for run_name in filtered["run_name"].unique():
    cfg = Config.load_from_file(name, str(base_path)+"/")

    set_random_seed(cfg.initial_seed)
    model = get_model(cfg.model_name, cfg.model_params).to(device)
    model.apply(init_weights)

    initial_weights = _parameters_to_vector(model) #initlal weights determined by the random seed initialization

    # Loading the trained parameters
    params_path = base_path / f"Model_{name}.pt"
    try:
        r = torch.load(params_path, map_location="cpu")
    except FileNotFoundError:
        print(f"Missing: {name}!")
        continue
    steady_state_params = r["model_state_dict"]
   
    # initialize a model with the same architecture and converged parameters as in the run
    model.load_state_dict(steady_state_params)

    # train a full GD until convergence
    quench_optimizer = get_opt("GD", _optimizer_params(experiment_cfg.quench_optimizer_params, experiment_cfg.quench_lr), model.parameters())
    final_loss = float("nan")
    final_gradient_norm = float("nan")
    for _ in range(experiment_cfg.quench_steps):
        final_loss, final_gradient_norm = _gd_step(model, task, full_data, quench_optimizer, loss_weights)
        if (
            experiment_cfg.quench_tolerance > 0
            and final_gradient_norm <= experiment_cfg.quench_tolerance
        ):
            break

    inherent_parameters = _parameters_to_vector(model)
    with torch.no_grad():
        final_loss = float(_full_batch_loss(model, task, full_data, loss_weights).cpu())

    # run all the comparison metrics (overlap etc) and save them
    result = {
        "temperature": experiment_cfg.temperature,
        "equilibration_steps": experiment_cfg.equilibration_steps,
        "quench_steps_requested": experiment_cfg.quench_steps,
        "final_loss": final_loss,
        "final_gradient_norm": final_gradient_norm,
        "overlap_q": cosine_overlap(equilibrated_parameters, inherent_parameters),
        "initial_to_inherent_overlap": cosine_overlap(initial_parameters, inherent_parameters),
        "initial_parameters": initial_parameters.cpu(),
        "equilibrated_parameters": equilibrated_parameters.cpu(),
        "inherent_parameters": inherent_parameters.cpu(),
        "equilibrated_model_state_dict": equilibrated_model_state_dict,
        "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    }

    #delete when not needed to save memory
    del r 
    gc.collect()

# plot the metrics as a funciton of the temperature.


# Next steps: add a comparison of Hessian structure.


SyntaxError: invalid syntax (784069727.py, line 11)